# Mixed-Type Self-Attention trên dataset v2 (bản cuối)

Kiến trúc và siêu tham số giữ nguyên bản 07b.

Dataset: pipeline v2 dựng lại từ raw (`02_artifacts/22_v2_pipeline`), split cấp cửa sổ phân tầng theo scenario. Một lần chạy (seed 42), đủ feature, không có tuỳ chọn bỏ feature hay ablation. Model selection và threshold chỉ dùng VAL; TEST chỉ chạy một lần sau khi checkpoint và threshold đã chốt.

In [ ]:
import os, sys, json, shutil
from pathlib import Path
import torch

print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))


## Cấu hình
Tên run cố định: `mixed_v2`.

In [ ]:
RUN_NAME = 'mixed_v2'
print('RUN_NAME =', RUN_NAME)


## 1. Resolve Kaggle dataset


In [ ]:
from __future__ import annotations

import shutil
from pathlib import Path


def unwrap_same_name_folder(
    root: Path,
    name: str,
) -> Path:
    path = root / name

    if not path.exists():
        raise FileNotFoundError(
            f"Missing folder: {path}"
        )

    while (path / name).is_dir():
        path = path / name

    return path


def resolve_kaggle_dataset_root(
    *,
    input_root: str | Path = "/kaggle/input",
    working_root: str | Path = "/kaggle/working/optc_network_dl",
) -> Path:
    input_root = Path(input_root)

    manifests = list(
        input_root.rglob(
            "dataset_manifest.json"
        )
    )

    if not manifests:
        raise FileNotFoundError(
            "Cannot find dataset_manifest.json under /kaggle/input"
        )

    if len(manifests) > 1:
        raise RuntimeError(
            "Nhieu dataset_manifest.json duoc tim thay (vd. dataset LOSO co 3 holdout). "
            "Chi gan MOT dataset vao notebook:\n"
            + "\n".join(str(m) for m in manifests)
        )

    source_root = None

    for manifest in manifests:
        candidate = manifest.parent

        if (
            candidate
            / "build_config.json"
        ).exists():
            source_root = candidate
            break

    if source_root is None:
        raise RuntimeError(
            "Found dataset_manifest.json but not build_config.json"
        )

    working_root = Path(working_root)

    if working_root.exists():
        shutil.rmtree(working_root)

    working_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    for filename in (
        "dataset_manifest.json",
        "build_config.json",
        "README_DATASET.md",
    ):
        src = source_root / filename

        if src.exists():
            (
                working_root / filename
            ).symlink_to(src)

    for folder in (
        "shards",
        "split",
        "preprocessing",
        "integrity",
    ):
        real_folder = unwrap_same_name_folder(
            source_root,
            folder,
        )

        (
            working_root / folder
        ).symlink_to(
            real_folder,
            target_is_directory=True,
        )

    required = [
        "dataset_manifest.json",
        "build_config.json",
        "split/network_sample_manifest.csv",
        "preprocessing/network_category_maps.json",
        "preprocessing/network_numeric_stats.json",
        "preprocessing/network_preprocess_spec.json",
        "preprocessing/network_preprocess_fingerprint.json",
        "shards/train",
        "shards/val",
        "shards/test",
    ]

    for relative in required:
        path = working_root / relative

        print(
            f"{'PASS' if path.exists() else 'FAIL':4s}",
            relative,
        )

        if not path.exists():
            raise FileNotFoundError(path)

    return working_root


DATASET_ROOT = resolve_kaggle_dataset_root()
print('\nDATASET_ROOT =', DATASET_ROOT)
manifest = json.loads((DATASET_ROOT/'dataset_manifest.json').read_text())
print(json.dumps(manifest['split_stats'], indent=2))


## 2. Write model/training modules


In [ ]:
WORK_CODE = Path('/kaggle/working/step05e_code')
WORK_CODE.mkdir(parents=True, exist_ok=True)
(WORK_CODE/'05e_attention_audit.py').write_text('from __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom network_dl_dataset_v1_1 import (\n    PackedNetworkDataset,\n    collate_sequence_batch,\n)\nfrom mixed_type_attention_model import (\n    MixedTypeModelConfig,\n    MixedTypeSelfAttentionClassifier,\n)\n\n\ndef load_json(path: Path):\n    return json.loads(path.read_text(encoding="utf-8"))\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--dataset-root", required=True)\n    parser.add_argument("--model-dir", required=True)\n    parser.add_argument(\n        "--split",\n        default="val",\n        choices=["train", "val", "test"],\n    )\n    parser.add_argument("--positive-samples", type=int, default=8)\n    parser.add_argument("--negative-samples", type=int, default=8)\n    args = parser.parse_args()\n\n    dataset_root = Path(args.dataset_root)\n    model_dir = Path(args.model_dir)\n\n    config = load_json(\n        model_dir / "config.json"\n    )\n\n    spec = load_json(\n        dataset_root\n        / "preprocessing"\n        / "network_preprocess_spec.json"\n    )\n    maps = load_json(\n        dataset_root\n        / "preprocessing"\n        / "network_category_maps.json"\n    )\n\n    cat_fields = list(\n        spec["active_categorical_features"]\n    )\n    num_fields = list(\n        spec["active_numerical_features"]\n    )\n\n    cards = [\n        max(\n            int(v)\n            for v in maps["fields"][field]["mapping"].values()\n        ) + 1\n        for field in cat_fields\n    ]\n\n    field_names = (\n        cat_fields\n        + num_fields\n        + ["has_missed_bytes"]\n        + ["delta_t_log1p"]\n    )\n\n    model_config = MixedTypeModelConfig(\n        categorical_cardinalities=cards,\n        numerical_dim=len(num_fields),\n        binary_dim=1,\n        max_events=int(config["max_events"]),\n        d_model=int(config["d_model"]),\n        num_heads=int(config["num_heads"]),\n        feature_layers=int(config["feature_layers"]),\n        temporal_layers=int(config["temporal_layers"]),\n        ff_dim=int(config["ff_dim"]),\n        dropout=float(config["dropout"]),\n    )\n\n    device = torch.device(\n        "cuda" if torch.cuda.is_available() else "cpu"\n    )\n\n    model = MixedTypeSelfAttentionClassifier(\n        model_config\n    ).to(device)\n\n    checkpoint = torch.load(\n        model_dir / "best_model.pt",\n        map_location=device,\n        weights_only=False,\n    )\n\n    model.load_state_dict(\n        checkpoint["model_state_dict"]\n    )\n    model.eval()\n\n    ds = PackedNetworkDataset(\n        dataset_root,\n        args.split,\n        cache_shards=1,\n    )\n\n    pos_indices = [\n        i\n        for i, row in enumerate(ds.rows)\n        if row["label"] == 1\n    ][:args.positive_samples]\n\n    neg_indices = [\n        i\n        for i, row in enumerate(ds.rows)\n        if row["label"] == 0\n    ][:args.negative_samples]\n\n    selected = (\n        [("positive", i) for i in pos_indices]\n        + [("negative", i) for i in neg_indices]\n    )\n\n    feature_received = {\n        "positive": np.zeros(\n            len(field_names),\n            dtype=np.float64,\n        ),\n        "negative": np.zeros(\n            len(field_names),\n            dtype=np.float64,\n        ),\n    }\n\n    feature_sample_counts = {\n        "positive": 0,\n        "negative": 0,\n    }\n\n    temporal_entropy = {\n        "positive": [],\n        "negative": [],\n    }\n\n    samples = []\n\n    with torch.no_grad():\n        for group, index in selected:\n            item = ds[index]\n            batch = collate_sequence_batch([item])\n\n            result = model(\n                x_cat=batch["x_cat"].to(device),\n                x_num=batch["x_num"].to(device),\n                x_num_missing=\n                    batch["x_num_missing"].to(device),\n                x_bin=batch["x_bin"].to(device),\n                delta_t_log1p=\n                    batch["delta_t_log1p"].to(device),\n                attention_mask=\n                    batch["attention_mask"].to(device),\n                return_attention=True,\n            )\n\n            prob = float(\n                torch.sigmoid(\n                    result["logits"]\n                )[0].cpu().item()\n            )\n\n            fa = result["feature_attention"]\n\n            if fa is not None:\n                # [events, heads, query_feature, key_feature]\n                # Average heads + queries + events:\n                # how much attention was received by each feature token.\n                received = (\n                    fa.float()\n                    .mean(dim=1)\n                    .mean(dim=1)\n                    .mean(dim=0)\n                    .cpu()\n                    .numpy()\n                )\n\n                feature_received[group] += received\n                feature_sample_counts[group] += 1\n\n            ta = result["temporal_attention"]\n\n            if ta is not None:\n                length = int(batch["lengths"][0])\n\n                matrix = (\n                    ta[0, :, :length, :length]\n                    .float()\n                    .mean(dim=0)\n                )\n\n                received_t = matrix.mean(dim=0)\n                received_t = (\n                    received_t\n                    / received_t.sum().clamp_min(1e-12)\n                )\n\n                entropy = float(\n                    -(\n                        received_t\n                        * torch.log(\n                            received_t.clamp_min(1e-12)\n                        )\n                    )\n                    .sum()\n                    .cpu()\n                    .item()\n                )\n\n                temporal_entropy[group].append(entropy)\n\n            samples.append({\n                "sample_id": item["sample_id"],\n                "group": group,\n                "label": int(item["label"]),\n                "probability": prob,\n                "event_count": int(item["event_count"]),\n            })\n\n    feature_summary = {}\n\n    for group in ("positive", "negative"):\n        divisor = max(\n            1,\n            feature_sample_counts[group],\n        )\n\n        scores = feature_received[group] / divisor\n\n        pairs = sorted(\n            zip(\n                field_names,\n                scores.tolist(),\n            ),\n            key=lambda x: -x[1],\n        )\n\n        feature_summary[group] = [\n            {\n                "field": name,\n                "mean_attention_received": float(score),\n            }\n            for name, score in pairs\n        ]\n\n    output = {\n        "split": args.split,\n        "warning": (\n            "Attention weights are diagnostic internal weights, "\n            "not causal feature importance."\n        ),\n        "feature_attention_received": feature_summary,\n        "temporal_attention_entropy": {\n            group: {\n                "n": len(values),\n                "mean": (\n                    float(np.mean(values))\n                    if values\n                    else None\n                ),\n                "values": values,\n            }\n            for group, values in temporal_entropy.items()\n        },\n        "samples": samples,\n    }\n\n    output_path = (\n        model_dir\n        / "attention_audit.json"\n    )\n\n    output_path.write_text(\n        json.dumps(\n            output,\n            indent=2,\n            ensure_ascii=False,\n        ),\n        encoding="utf-8",\n    )\n\n    print(f"[CREATED] {output_path}")\n\n    for group in ("positive", "negative"):\n        print()\n        print(\n            f"[TOP FEATURE ATTENTION — {group.upper()}]"\n        )\n\n        for item in feature_summary[group][:8]:\n            print(\n                f"  {item[\'field\']:24s} "\n                f"{item[\'mean_attention_received\']:.6f}"\n            )\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
(WORK_CODE/'05e_train_mixed_type_attention.py').write_text('from __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport random\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom sklearn.metrics import (\n    average_precision_score,\n    confusion_matrix,\n    f1_score,\n    precision_recall_curve,\n    precision_score,\n    recall_score,\n    roc_auc_score,\n)\nfrom torch.utils.data import DataLoader\n\nfrom network_dl_dataset_v1_1 import (\n    PackedNetworkDataset,\n    ShardAwareBatchSampler,\n    collate_sequence_batch,\n)\nfrom mixed_type_attention_model import (\n    MixedTypeModelConfig,\n    MixedTypeSelfAttentionClassifier,\n)\n\n\nSCRIPT_VERSION = "step05e-mixed-type-self-attention-v1.0"\n\n\ndef load_json(path: Path) -> Any:\n    return json.loads(path.read_text(encoding="utf-8"))\n\n\ndef dump_json(path: Path, obj: Any) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(\n        json.dumps(obj, indent=2, ensure_ascii=False),\n        encoding="utf-8",\n    )\n\n\ndef seed_everything(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef categorical_cardinalities(\n    dataset_root: Path,\n) -> tuple[list[str], list[int]]:\n    spec = load_json(\n        dataset_root\n        / "preprocessing"\n        / "network_preprocess_spec.json"\n    )\n    maps = load_json(\n        dataset_root\n        / "preprocessing"\n        / "network_category_maps.json"\n    )\n\n    fields = list(\n        spec["active_categorical_features"]\n    )\n\n    cards = [\n        max(\n            int(v)\n            for v in maps["fields"][field]["mapping"].values()\n        ) + 1\n        for field in fields\n    ]\n\n    return fields, cards\n\n\ndef numerical_fields(\n    dataset_root: Path,\n) -> list[str]:\n    spec = load_json(\n        dataset_root\n        / "preprocessing"\n        / "network_preprocess_spec.json"\n    )\n    return list(spec["active_numerical_features"])\n\n\ndef make_loader(\n    dataset: PackedNetworkDataset,\n    *,\n    batch_size: int,\n    shuffle: bool,\n    seed: int,\n    num_workers: int,\n    pin_memory: bool,\n):\n    sampler = ShardAwareBatchSampler(\n        dataset,\n        batch_size,\n        shuffle=shuffle,\n        seed=seed,\n        drop_last=False,\n    )\n\n    loader = DataLoader(\n        dataset,\n        batch_sampler=sampler,\n        num_workers=num_workers,\n        pin_memory=pin_memory,\n        persistent_workers=(num_workers > 0),\n        collate_fn=collate_sequence_batch,\n    )\n\n    return loader, sampler\n\n\ndef move_batch(\n    batch: dict[str, Any],\n    device: torch.device,\n) -> dict[str, Any]:\n    result = dict(batch)\n\n    for key in (\n        "x_cat",\n        "x_num",\n        "x_num_missing",\n        "x_bin",\n        "delta_t_log1p",\n        "attention_mask",\n        "labels",\n    ):\n        result[key] = batch[key].to(\n            device,\n            non_blocking=True,\n        )\n\n    return result\n\n\ndef binary_metrics(\n    y_true: np.ndarray,\n    y_prob: np.ndarray,\n    threshold: float,\n) -> dict[str, Any]:\n    y_pred = (y_prob >= threshold).astype(np.uint8)\n\n    tn, fp, fn, tp = confusion_matrix(\n        y_true,\n        y_pred,\n        labels=[0, 1],\n    ).ravel()\n\n    precision = precision_score(\n        y_true,\n        y_pred,\n        zero_division=0,\n    )\n    recall = recall_score(\n        y_true,\n        y_pred,\n        zero_division=0,\n    )\n    f1 = f1_score(\n        y_true,\n        y_pred,\n        zero_division=0,\n    )\n    fpr = fp / (fp + tn) if (fp + tn) else 0.0\n\n    result = {\n        "threshold": float(threshold),\n        "precision": float(precision),\n        "recall": float(recall),\n        "f1": float(f1),\n        "fpr": float(fpr),\n        "tn": int(tn),\n        "fp": int(fp),\n        "fn": int(fn),\n        "tp": int(tp),\n        "pr_auc": float(\n            average_precision_score(y_true, y_prob)\n        ),\n    }\n\n    result["roc_auc"] = (\n        float(roc_auc_score(y_true, y_prob))\n        if len(np.unique(y_true)) > 1\n        else None\n    )\n\n    return result\n\n\ndef best_f1_threshold(\n    y_true: np.ndarray,\n    y_prob: np.ndarray,\n):\n    precision, recall, thresholds = precision_recall_curve(\n        y_true,\n        y_prob,\n    )\n\n    if len(thresholds) == 0:\n        threshold = 0.5\n        return threshold, binary_metrics(\n            y_true,\n            y_prob,\n            threshold,\n        )\n\n    p = precision[:-1]\n    r = recall[:-1]\n    denom = p + r\n    f1 = np.divide(\n        2 * p * r,\n        denom,\n        out=np.zeros_like(denom),\n        where=denom > 0,\n    )\n    idx = int(np.nanargmax(f1))\n    threshold = float(thresholds[idx])\n\n    return threshold, binary_metrics(\n        y_true,\n        y_prob,\n        threshold,\n    )\n\n\n@torch.no_grad()\ndef predict(\n    model: nn.Module,\n    loader: DataLoader,\n    *,\n    device: torch.device,\n    amp: bool,\n    phase: str,\n    progress_every: int = 100,\n    max_batches: int | None = None,\n):\n    model.eval()\n\n    probs = []\n    labels = []\n    sample_ids = []\n\n    total = len(loader)\n\n    for step, raw_batch in enumerate(\n        loader,\n        start=1,\n    ):\n        if (\n            max_batches is not None\n            and step > max_batches\n        ):\n            break\n\n        batch = move_batch(raw_batch, device)\n\n        with torch.autocast(\n            device_type=device.type,\n            dtype=torch.float16,\n            enabled=(amp and device.type == "cuda"),\n        ):\n            logits = model(\n                x_cat=batch["x_cat"],\n                x_num=batch["x_num"],\n                x_num_missing=batch["x_num_missing"],\n                x_bin=batch["x_bin"],\n                delta_t_log1p=batch["delta_t_log1p"],\n                attention_mask=batch["attention_mask"],\n            )\n\n        probs.append(\n            torch.sigmoid(logits).cpu().numpy()\n        )\n        labels.append(\n            batch["labels"].cpu().numpy()\n        )\n        sample_ids.extend(raw_batch["sample_ids"])\n\n        if (\n            progress_every > 0\n            and (\n                step % progress_every == 0\n                or step == total\n            )\n        ):\n            print(\n                f"[{phase}] batch={step}/{total}"\n            )\n\n    return (\n        np.concatenate(labels),\n        np.concatenate(probs),\n        sample_ids,\n    )\n\n\ndef save_predictions(\n    path: Path,\n    *,\n    sample_ids,\n    y_true,\n    y_prob,\n    threshold,\n):\n    y_pred = (y_prob >= threshold).astype(np.uint8)\n\n    with path.open(\n        "w",\n        newline="",\n        encoding="utf-8",\n    ) as f:\n        writer = csv.writer(f)\n        writer.writerow([\n            "sample_id",\n            "label",\n            "probability",\n            "prediction",\n        ])\n\n        for sid, y, p, pred in zip(\n            sample_ids,\n            y_true,\n            y_prob,\n            y_pred,\n        ):\n            writer.writerow([\n                sid,\n                int(y),\n                float(p),\n                int(pred),\n            ])\n\n\ndef save_curves(\n    output_dir: Path,\n    *,\n    y_true,\n    y_prob,\n    prefix: str,\n):\n    import matplotlib.pyplot as plt\n    from sklearn.metrics import (\n        PrecisionRecallDisplay,\n        RocCurveDisplay,\n    )\n\n    fig, ax = plt.subplots()\n    PrecisionRecallDisplay.from_predictions(\n        y_true,\n        y_prob,\n        ax=ax,\n    )\n    ax.set_title(f"{prefix.upper()} Precision-Recall")\n    fig.tight_layout()\n    fig.savefig(\n        output_dir / f"{prefix}_pr_curve.png",\n        dpi=150,\n    )\n    plt.close(fig)\n\n    if len(np.unique(y_true)) > 1:\n        fig, ax = plt.subplots()\n        RocCurveDisplay.from_predictions(\n            y_true,\n            y_prob,\n            ax=ax,\n        )\n        ax.set_title(f"{prefix.upper()} ROC")\n        fig.tight_layout()\n        fig.savefig(\n            output_dir / f"{prefix}_roc_curve.png",\n            dpi=150,\n        )\n        plt.close(fig)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n\n    parser.add_argument("--dataset-root", required=True)\n    parser.add_argument("--output-dir", required=True)\n\n    parser.add_argument("--epochs", type=int, default=8)\n    parser.add_argument("--batch-size", type=int, default=16)\n    parser.add_argument("--grad-accum", type=int, default=2)\n\n    parser.add_argument("--d-model", type=int, default=64)\n    parser.add_argument("--heads", type=int, default=4)\n    parser.add_argument("--feature-layers", type=int, default=1)\n    parser.add_argument("--temporal-layers", type=int, default=2)\n    parser.add_argument("--ff-dim", type=int, default=128)\n    parser.add_argument("--dropout", type=float, default=0.2)\n\n    parser.add_argument("--lr", type=float, default=3e-4)\n    parser.add_argument("--weight-decay", type=float, default=1e-4)\n    parser.add_argument("--grad-clip", type=float, default=1.0)\n    parser.add_argument("--patience", type=int, default=3)\n\n    parser.add_argument("--num-workers", type=int, default=2)\n    parser.add_argument("--cache-shards", type=int, default=1)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--eval-progress-every", type=int, default=100)\n\n    parser.add_argument("--no-amp", action="store_true")\n    parser.add_argument("--max-train-batches", type=int, default=None)\n    parser.add_argument("--max-eval-batches", type=int, default=None)\n\n    args = parser.parse_args()\n\n    if args.d_model % args.heads != 0:\n        raise ValueError(\n            "--d-model must be divisible by --heads"\n        )\n    if args.grad_accum < 1:\n        raise ValueError("--grad-accum must be >= 1")\n\n    dataset_root = Path(args.dataset_root)\n    output_dir = Path(args.output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    seed_everything(args.seed)\n\n    manifest = load_json(\n        dataset_root / "dataset_manifest.json"\n    )\n    build_config = manifest["build_config"]\n    max_events = int(build_config["max_events"])\n\n    train_stats = manifest["split_stats"]["train"]\n    train_positive = int(train_stats["positive_samples"])\n    train_negative = int(train_stats["negative_samples"])\n\n    pos_weight_value = (\n        train_negative / train_positive\n    )\n\n    cat_fields, cat_cards = categorical_cardinalities(\n        dataset_root\n    )\n    num_fields = numerical_fields(dataset_root)\n\n    device = torch.device(\n        "cuda" if torch.cuda.is_available() else "cpu"\n    )\n    amp = (\n        not args.no_amp\n        and device.type == "cuda"\n    )\n\n    print("=" * 104)\n    print("STEP 05E — Mixed-Type Self-Attention")\n    print("=" * 104)\n    print(f"[DATASET] {dataset_root}")\n    print(\n        f"[TRAIN] negative={train_negative:,} | "\n        f"positive={train_positive:,} | "\n        f"pos_weight={pos_weight_value:.6f}"\n    )\n    print(f"[CAT] {list(zip(cat_fields, cat_cards))}")\n    print(f"[NUM] {num_fields}")\n    print(f"[MAX EVENTS] {max_events}")\n    print(f"[DEVICE] {device} | amp={amp}")\n\n    train_ds = PackedNetworkDataset(\n        dataset_root,\n        "train",\n        cache_shards=args.cache_shards,\n    )\n    val_ds = PackedNetworkDataset(\n        dataset_root,\n        "val",\n        cache_shards=args.cache_shards,\n    )\n    test_ds = PackedNetworkDataset(\n        dataset_root,\n        "test",\n        cache_shards=args.cache_shards,\n    )\n    pin_memory = device.type == "cuda"\n\n    train_loader, train_sampler = make_loader(\n        train_ds,\n        batch_size=args.batch_size,\n        shuffle=True,\n        seed=args.seed,\n        num_workers=args.num_workers,\n        pin_memory=pin_memory,\n    )\n    val_loader, _ = make_loader(\n        val_ds,\n        batch_size=args.batch_size,\n        shuffle=False,\n        seed=args.seed,\n        num_workers=args.num_workers,\n        pin_memory=pin_memory,\n    )\n    test_loader, _ = make_loader(\n        test_ds,\n        batch_size=args.batch_size,\n        shuffle=False,\n        seed=args.seed,\n        num_workers=args.num_workers,\n        pin_memory=pin_memory,\n    )\n\n    model_config = MixedTypeModelConfig(\n        categorical_cardinalities=cat_cards,\n        numerical_dim=len(num_fields),\n        binary_dim=1,\n        max_events=max_events,\n        d_model=args.d_model,\n        num_heads=args.heads,\n        feature_layers=args.feature_layers,\n        temporal_layers=args.temporal_layers,\n        ff_dim=args.ff_dim,\n        dropout=args.dropout,\n    )\n\n    model = MixedTypeSelfAttentionClassifier(\n        model_config\n    ).to(device)\n\n    trainable = sum(\n        p.numel()\n        for p in model.parameters()\n        if p.requires_grad\n    )\n\n    print(f"[PARAMS] {trainable:,}")\n    print(\n        f"[ATTN] d_model={args.d_model} | heads={args.heads} | "\n        f"feature_layers={args.feature_layers} | "\n        f"temporal_layers={args.temporal_layers}"\n    )\n    print(\n        f"[BATCH] physical={args.batch_size} | "\n        f"grad_accum={args.grad_accum} | "\n        f"effective≈{args.batch_size * args.grad_accum}"\n    )\n\n    pos_weight = torch.tensor(\n        pos_weight_value,\n        dtype=torch.float32,\n        device=device,\n    )\n\n    criterion = nn.BCEWithLogitsLoss(\n        pos_weight=pos_weight\n    )\n\n    optimizer = torch.optim.AdamW(\n        model.parameters(),\n        lr=args.lr,\n        weight_decay=args.weight_decay,\n    )\n\n    scaler = torch.amp.GradScaler(\n        "cuda",\n        enabled=amp,\n    )\n\n    config = {\n        "script_version": SCRIPT_VERSION,\n        "dataset_root": str(dataset_root),\n        "dataset_version": build_config.get("dataset_version"),\n        "dataset_preprocess_fingerprint":\n            (manifest["source"].get("step04c_preprocess_fingerprint") or manifest["source"]["step05f_c_preprocess_fingerprint"]),\n        "seed": args.seed,\n        "epochs": args.epochs,\n        "physical_batch_size": args.batch_size,\n        "gradient_accumulation": args.grad_accum,\n        "effective_batch_size": args.batch_size * args.grad_accum,\n        "d_model": args.d_model,\n        "num_heads": args.heads,\n        "feature_layers": args.feature_layers,\n        "temporal_layers": args.temporal_layers,\n        "ff_dim": args.ff_dim,\n        "dropout": args.dropout,\n        "max_events": max_events,\n        "lr": args.lr,\n        "weight_decay": args.weight_decay,\n        "grad_clip": args.grad_clip,\n        "patience": args.patience,\n        "num_workers": args.num_workers,\n        "amp": amp,\n        "loss": "BCEWithLogitsLoss",\n        "train_pos_weight": pos_weight_value,\n        "categorical_fields": cat_fields,\n        "categorical_cardinalities": cat_cards,\n        "numerical_fields": num_fields,\n        "binary_fields": ["has_missed_bytes"],\n        "delta_time_field": "delta_t_log1p",\n        "feature_token_count":\n            len(cat_fields) + len(num_fields) + 2,\n        "event_pooling": "mean over feature-token outputs",\n        "temporal_position":\n            "learned absolute position embedding",\n        "window_pooling": "masked mean over event outputs",\n        "causal_mask": False,\n        "early_stopping_metric": "VAL PR-AUC",\n        "threshold_policy": "maximize F1 on VAL only",\n        "test_policy":\n            "evaluate once after best checkpoint + VAL threshold freeze",\n        "trainable_parameters": trainable,\n    }\n\n    dump_json(\n        output_dir / "config.json",\n        config,\n    )\n\n    best_pr_auc = -1.0\n    best_epoch = -1\n    stale_epochs = 0\n    history = []\n\n    best_path = output_dir / "best_model.pt"\n\n    for epoch in range(1, args.epochs + 1):\n        train_sampler.set_epoch(epoch)\n        model.train()\n        optimizer.zero_grad(set_to_none=True)\n\n        running_loss = 0.0\n        seen_batches = 0\n        optimizer_steps = 0\n        started = time.time()\n\n        for step, raw_batch in enumerate(\n            train_loader,\n            start=1,\n        ):\n            if (\n                args.max_train_batches is not None\n                and step > args.max_train_batches\n            ):\n                break\n\n            batch = move_batch(raw_batch, device)\n\n            with torch.autocast(\n                device_type=device.type,\n                dtype=torch.float16,\n                enabled=amp,\n            ):\n                logits = model(\n                    x_cat=batch["x_cat"],\n                    x_num=batch["x_num"],\n                    x_num_missing=batch["x_num_missing"],\n                    x_bin=batch["x_bin"],\n                    delta_t_log1p=batch["delta_t_log1p"],\n                    attention_mask=batch["attention_mask"],\n                )\n\n                raw_loss = criterion(\n                    logits,\n                    batch["labels"],\n                )\n                loss = raw_loss / args.grad_accum\n\n            scaler.scale(loss).backward()\n\n            running_loss += float(raw_loss.item())\n            seen_batches += 1\n\n            should_step = (\n                step % args.grad_accum == 0\n                or step == len(train_loader)\n            )\n\n            if should_step:\n                scaler.unscale_(optimizer)\n\n                torch.nn.utils.clip_grad_norm_(\n                    model.parameters(),\n                    args.grad_clip,\n                )\n\n                scaler.step(optimizer)\n                scaler.update()\n                optimizer.zero_grad(set_to_none=True)\n                optimizer_steps += 1\n\n            if step % 200 == 0:\n                print(\n                    f"[EPOCH {epoch}] "\n                    f"batch={step}/{len(train_loader)} | "\n                    f"loss={running_loss/seen_batches:.6f} | "\n                    f"opt_steps={optimizer_steps}"\n                )\n\n        train_loss = running_loss / max(1, seen_batches)\n\n        print(\n            f"[EPOCH {epoch}] training complete -> validating..."\n        )\n\n        val_true, val_prob, _ = predict(\n            model,\n            val_loader,\n            device=device,\n            amp=amp,\n            phase="VAL",\n            progress_every=args.eval_progress_every,\n            max_batches=args.max_eval_batches,\n        )\n\n        val_pr_auc = float(\n            average_precision_score(\n                val_true,\n                val_prob,\n            )\n        )\n\n        val_threshold, val_metrics = best_f1_threshold(\n            val_true,\n            val_prob,\n        )\n\n        elapsed = time.time() - started\n\n        row = {\n            "epoch": epoch,\n            "train_loss": train_loss,\n            "val_pr_auc": val_pr_auc,\n            "val_best_f1_threshold": val_threshold,\n            "val_precision": val_metrics["precision"],\n            "val_recall": val_metrics["recall"],\n            "val_f1": val_metrics["f1"],\n            "val_fpr": val_metrics["fpr"],\n            "elapsed_seconds": elapsed,\n        }\n        history.append(row)\n\n        print(\n            f"[EPOCH {epoch}] "\n            f"loss={train_loss:.6f} | "\n            f"val_PR_AUC={val_pr_auc:.6f} | "\n            f"val_F1={val_metrics[\'f1\']:.6f} | "\n            f"thr={val_threshold:.8f} | "\n            f"{elapsed/60:.1f} min"\n        )\n\n        if val_pr_auc > best_pr_auc + 1e-8:\n            best_pr_auc = val_pr_auc\n            best_epoch = epoch\n            stale_epochs = 0\n\n            torch.save(\n                {\n                    "model_state_dict": model.state_dict(),\n                    "model_config": model_config.__dict__,\n                    "training_config": config,\n                    "best_epoch": best_epoch,\n                    "best_val_pr_auc": best_pr_auc,\n                },\n                best_path,\n            )\n\n            print(\n                f"[CHECKPOINT] best_model.pt epoch={epoch}"\n            )\n        else:\n            stale_epochs += 1\n\n        if stale_epochs >= args.patience:\n            print(\n                "[EARLY STOP] "\n                f"no PR-AUC improvement for {stale_epochs} epochs"\n            )\n            break\n\n    history_path = output_dir / "history.csv"\n\n    with history_path.open(\n        "w",\n        newline="",\n        encoding="utf-8",\n    ) as f:\n        writer = csv.DictWriter(\n            f,\n            fieldnames=list(history[0].keys()),\n        )\n        writer.writeheader()\n        writer.writerows(history)\n\n    checkpoint = torch.load(\n        best_path,\n        map_location=device,\n        weights_only=False,\n    )\n    model.load_state_dict(\n        checkpoint["model_state_dict"]\n    )\n\n    print(\n        "[FINAL] best checkpoint loaded -> selecting threshold on VAL..."\n    )\n\n    val_true, val_prob, val_ids = predict(\n        model,\n        val_loader,\n        device=device,\n        amp=amp,\n        phase="VAL-FINAL",\n        progress_every=args.eval_progress_every,\n        max_batches=args.max_eval_batches,\n    )\n\n    threshold, val_metrics = best_f1_threshold(\n        val_true,\n        val_prob,\n    )\n\n    dump_json(\n        output_dir / "threshold.json",\n        {\n            "selected_on": "val",\n            "policy": "maximize F1",\n            "threshold": threshold,\n            "best_epoch": best_epoch,\n            "best_val_pr_auc": best_pr_auc,\n            "val_metrics_at_threshold": val_metrics,\n        },\n    )\n\n    save_predictions(\n        output_dir / "val_predictions.csv",\n        sample_ids=val_ids,\n        y_true=val_true,\n        y_prob=val_prob,\n        threshold=threshold,\n    )\n\n    save_curves(\n        output_dir,\n        y_true=val_true,\n        y_prob=val_prob,\n        prefix="val",\n    )\n\n    print(\n        "[FINAL] VAL threshold frozen -> evaluating TEST once..."\n    )\n\n    test_true, test_prob, test_ids = predict(\n        model,\n        test_loader,\n        device=device,\n        amp=amp,\n        phase="TEST",\n        progress_every=args.eval_progress_every,\n        max_batches=args.max_eval_batches,\n    )\n\n    test_metrics = binary_metrics(\n        test_true,\n        test_prob,\n        threshold,\n    )\n\n    save_predictions(\n        output_dir / "test_predictions.csv",\n        sample_ids=test_ids,\n        y_true=test_true,\n        y_prob=test_prob,\n        threshold=threshold,\n    )\n\n    save_curves(\n        output_dir,\n        y_true=test_true,\n        y_prob=test_prob,\n        prefix="test",\n    )\n\n    final_metrics = {\n        "script_version": SCRIPT_VERSION,\n        "best_epoch": best_epoch,\n        "best_val_pr_auc": best_pr_auc,\n        "threshold_selected_on_val": threshold,\n        "val": val_metrics,\n        "test": test_metrics,\n    }\n\n    dump_json(\n        output_dir / "metrics.json",\n        final_metrics,\n    )\n\n    print()\n    print("=" * 104)\n    print("[SUCCESS] Mixed-Type Self-Attention complete")\n    print(f"[BEST EPOCH] {best_epoch}")\n    print(\n        f"[VAL] PR-AUC={val_metrics[\'pr_auc\']:.6f} | "\n        f"F1={val_metrics[\'f1\']:.6f} | "\n        f"Recall={val_metrics[\'recall\']:.6f} | "\n        f"FPR={val_metrics[\'fpr\']:.8f}"\n    )\n    print(\n        f"[TEST] PR-AUC={test_metrics[\'pr_auc\']:.6f} | "\n        f"F1={test_metrics[\'f1\']:.6f} | "\n        f"Recall={test_metrics[\'recall\']:.6f} | "\n        f"FPR={test_metrics[\'fpr\']:.8f}"\n    )\n    print(f"[OUTPUT] {output_dir}")\n    print("=" * 104)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
(WORK_CODE/'mixed_type_attention_model.py').write_text('from __future__ import annotations\n\nfrom dataclasses import dataclass\n\nimport torch\nimport torch.nn as nn\n\n\n@dataclass(frozen=True)\nclass MixedTypeModelConfig:\n    categorical_cardinalities: list[int]\n    numerical_dim: int\n    binary_dim: int = 1\n    max_events: int = 512\n    d_model: int = 64\n    num_heads: int = 4\n    feature_layers: int = 1\n    temporal_layers: int = 2\n    ff_dim: int = 128\n    dropout: float = 0.2\n\n\nclass SelfAttentionBlock(nn.Module):\n    def __init__(\n        self,\n        *,\n        d_model: int,\n        num_heads: int,\n        ff_dim: int,\n        dropout: float,\n    ):\n        super().__init__()\n\n        if d_model % num_heads != 0:\n            raise ValueError(\n                "d_model must be divisible by num_heads"\n            )\n\n        self.norm1 = nn.LayerNorm(d_model)\n        self.attn = nn.MultiheadAttention(\n            d_model,\n            num_heads,\n            dropout=dropout,\n            batch_first=True,\n        )\n        self.attn_dropout = nn.Dropout(dropout)\n\n        self.norm2 = nn.LayerNorm(d_model)\n        self.ff = nn.Sequential(\n            nn.Linear(d_model, ff_dim),\n            nn.GELU(),\n            nn.Dropout(dropout),\n            nn.Linear(ff_dim, d_model),\n            nn.Dropout(dropout),\n        )\n\n    def forward(\n        self,\n        x: torch.Tensor,\n        *,\n        key_padding_mask: torch.Tensor | None = None,\n        return_attention: bool = False,\n    ):\n        y = self.norm1(x)\n\n        attended, weights = self.attn(\n            y,\n            y,\n            y,\n            key_padding_mask=key_padding_mask,\n            need_weights=return_attention,\n            average_attn_weights=False,\n        )\n\n        x = x + self.attn_dropout(attended)\n        x = x + self.ff(self.norm2(x))\n\n        return x, weights\n\n\nclass MixedTypeEventEncoder(nn.Module):\n    TYPE_CAT = 0\n    TYPE_NUM = 1\n    TYPE_BIN = 2\n    TYPE_DELTA = 3\n\n    def __init__(\n        self,\n        *,\n        categorical_cardinalities: list[int],\n        numerical_dim: int,\n        binary_dim: int,\n        d_model: int,\n        num_heads: int,\n        num_layers: int,\n        ff_dim: int,\n        dropout: float,\n    ):\n        super().__init__()\n\n        self.categorical_dim = len(categorical_cardinalities)\n        self.numerical_dim = int(numerical_dim)\n        self.binary_dim = int(binary_dim)\n\n        if self.binary_dim != 1:\n            raise ValueError(\n                "Current baseline expects one binary field."\n            )\n\n        self.cat_embeddings = nn.ModuleList([\n            nn.Embedding(int(cardinality), d_model)\n            for cardinality in categorical_cardinalities\n        ])\n\n        self.num_weight = nn.Parameter(\n            torch.empty(self.numerical_dim, d_model)\n        )\n        self.num_bias = nn.Parameter(\n            torch.zeros(self.numerical_dim, d_model)\n        )\n        self.num_missing_vector = nn.Parameter(\n            torch.zeros(self.numerical_dim, d_model)\n        )\n\n        self.binary_embedding = nn.Embedding(2, d_model)\n\n        self.delta_weight = nn.Parameter(\n            torch.empty(d_model)\n        )\n        self.delta_bias = nn.Parameter(\n            torch.zeros(d_model)\n        )\n\n        self.total_feature_tokens = (\n            self.categorical_dim\n            + self.numerical_dim\n            + self.binary_dim\n            + 1\n        )\n\n        self.field_embedding = nn.Embedding(\n            self.total_feature_tokens,\n            d_model,\n        )\n        self.type_embedding = nn.Embedding(4, d_model)\n\n        field_type_ids = (\n            [self.TYPE_CAT] * self.categorical_dim\n            + [self.TYPE_NUM] * self.numerical_dim\n            + [self.TYPE_BIN] * self.binary_dim\n            + [self.TYPE_DELTA]\n        )\n\n        self.register_buffer(\n            "field_type_ids",\n            torch.tensor(field_type_ids, dtype=torch.long),\n            persistent=False,\n        )\n\n        self.blocks = nn.ModuleList([\n            SelfAttentionBlock(\n                d_model=d_model,\n                num_heads=num_heads,\n                ff_dim=ff_dim,\n                dropout=dropout,\n            )\n            for _ in range(num_layers)\n        ])\n\n        self.output_norm = nn.LayerNorm(d_model)\n\n        nn.init.normal_(self.num_weight, mean=0.0, std=0.02)\n        nn.init.normal_(\n            self.num_missing_vector,\n            mean=0.0,\n            std=0.02,\n        )\n        nn.init.normal_(self.delta_weight, mean=0.0, std=0.02)\n\n    def build_feature_tokens(\n        self,\n        *,\n        x_cat: torch.Tensor,\n        x_num: torch.Tensor,\n        x_num_missing: torch.Tensor,\n        x_bin: torch.Tensor,\n        delta_t_log1p: torch.Tensor,\n    ) -> torch.Tensor:\n        cat_tokens = torch.stack(\n            [\n                emb(x_cat[:, i])\n                for i, emb in enumerate(self.cat_embeddings)\n            ],\n            dim=1,\n        )\n\n        num_tokens = (\n            x_num.unsqueeze(-1)\n            * self.num_weight.unsqueeze(0)\n            + self.num_bias.unsqueeze(0)\n        )\n\n        num_tokens = (\n            num_tokens\n            + x_num_missing.unsqueeze(-1)\n            * self.num_missing_vector.unsqueeze(0)\n        )\n\n        bin_token = self.binary_embedding(\n            x_bin[:, 0].long().clamp(0, 1)\n        ).unsqueeze(1)\n\n        delta_token = (\n            delta_t_log1p[:, 0].unsqueeze(-1)\n            * self.delta_weight.unsqueeze(0)\n            + self.delta_bias.unsqueeze(0)\n        ).unsqueeze(1)\n\n        tokens = torch.cat(\n            [\n                cat_tokens,\n                num_tokens,\n                bin_token,\n                delta_token,\n            ],\n            dim=1,\n        )\n\n        field_ids = torch.arange(\n            self.total_feature_tokens,\n            device=tokens.device,\n        )\n\n        tokens = (\n            tokens\n            + self.field_embedding(field_ids)[None, :, :]\n            + self.type_embedding(\n                self.field_type_ids\n            )[None, :, :]\n        )\n\n        return tokens\n\n    def forward(\n        self,\n        *,\n        x_cat: torch.Tensor,\n        x_num: torch.Tensor,\n        x_num_missing: torch.Tensor,\n        x_bin: torch.Tensor,\n        delta_t_log1p: torch.Tensor,\n        return_attention: bool = False,\n    ):\n        tokens = self.build_feature_tokens(\n            x_cat=x_cat,\n            x_num=x_num,\n            x_num_missing=x_num_missing,\n            x_bin=x_bin,\n            delta_t_log1p=delta_t_log1p,\n        )\n\n        last_attention = None\n\n        for i, block in enumerate(self.blocks):\n            tokens, weights = block(\n                tokens,\n                return_attention=(\n                    return_attention\n                    and i == len(self.blocks) - 1\n                ),\n            )\n            if weights is not None:\n                last_attention = weights\n\n        tokens = self.output_norm(tokens)\n\n        # Feature-level classification-style pooling.\n        event_embedding = tokens.mean(dim=1)\n\n        return event_embedding, last_attention\n\n\nclass MixedTypeSelfAttentionClassifier(nn.Module):\n    def __init__(\n        self,\n        config: MixedTypeModelConfig,\n    ):\n        super().__init__()\n        self.config = config\n\n        self.event_encoder = MixedTypeEventEncoder(\n            categorical_cardinalities=\n                config.categorical_cardinalities,\n            numerical_dim=\n                config.numerical_dim,\n            binary_dim=\n                config.binary_dim,\n            d_model=\n                config.d_model,\n            num_heads=\n                config.num_heads,\n            num_layers=\n                config.feature_layers,\n            ff_dim=\n                config.ff_dim,\n            dropout=\n                config.dropout,\n        )\n\n        self.position_embedding = nn.Embedding(\n            config.max_events,\n            config.d_model,\n        )\n\n        self.temporal_blocks = nn.ModuleList([\n            SelfAttentionBlock(\n                d_model=config.d_model,\n                num_heads=config.num_heads,\n                ff_dim=config.ff_dim,\n                dropout=config.dropout,\n            )\n            for _ in range(config.temporal_layers)\n        ])\n\n        self.temporal_norm = nn.LayerNorm(config.d_model)\n\n        self.classifier = nn.Sequential(\n            nn.Linear(config.d_model, config.d_model),\n            nn.GELU(),\n            nn.Dropout(config.dropout),\n            nn.Linear(config.d_model, 1),\n        )\n\n    def forward(\n        self,\n        *,\n        x_cat: torch.Tensor,\n        x_num: torch.Tensor,\n        x_num_missing: torch.Tensor,\n        x_bin: torch.Tensor,\n        delta_t_log1p: torch.Tensor,\n        attention_mask: torch.Tensor,\n        return_attention: bool = False,\n    ):\n        b, t, _ = x_cat.shape\n\n        if t > self.config.max_events:\n            raise ValueError(\n                f"T={t} exceeds max_events={self.config.max_events}"\n            )\n\n        valid = attention_mask.bool()\n        flat_valid = valid.reshape(-1)\n\n        flat_cat = x_cat.reshape(\n            b * t,\n            x_cat.shape[-1],\n        )[flat_valid]\n\n        flat_num = x_num.reshape(\n            b * t,\n            x_num.shape[-1],\n        )[flat_valid]\n\n        flat_missing = x_num_missing.reshape(\n            b * t,\n            x_num_missing.shape[-1],\n        )[flat_valid]\n\n        flat_bin = x_bin.reshape(\n            b * t,\n            x_bin.shape[-1],\n        )[flat_valid]\n\n        flat_delta = delta_t_log1p.reshape(\n            b * t,\n            1,\n        )[flat_valid]\n\n        event_valid, feature_attention = self.event_encoder(\n            x_cat=flat_cat,\n            x_num=flat_num,\n            x_num_missing=flat_missing,\n            x_bin=flat_bin,\n            delta_t_log1p=flat_delta,\n            return_attention=return_attention,\n        )\n\n        event_embeddings = torch.zeros(\n            b * t,\n            self.config.d_model,\n            device=x_cat.device,\n            dtype=event_valid.dtype,\n        )\n\n        event_embeddings[flat_valid] = event_valid\n\n        event_embeddings = event_embeddings.view(\n            b,\n            t,\n            self.config.d_model,\n        )\n\n        positions = torch.arange(\n            t,\n            device=x_cat.device,\n        )\n\n        temporal = (\n            event_embeddings\n            + self.position_embedding(positions)[None, :, :]\n        )\n\n        key_padding_mask = ~valid\n        last_temporal_attention = None\n\n        for i, block in enumerate(self.temporal_blocks):\n            temporal, weights = block(\n                temporal,\n                key_padding_mask=key_padding_mask,\n                return_attention=(\n                    return_attention\n                    and i == len(self.temporal_blocks) - 1\n                ),\n            )\n            if weights is not None:\n                last_temporal_attention = weights\n\n        temporal = self.temporal_norm(temporal)\n\n        mask_float = valid.unsqueeze(-1).to(temporal.dtype)\n\n        pooled = (\n            (temporal * mask_float).sum(dim=1)\n            / mask_float.sum(dim=1).clamp_min(1.0)\n        )\n\n        logits = self.classifier(pooled).squeeze(-1)\n\n        if return_attention:\n            return {\n                "logits": logits,\n                "feature_attention": feature_attention,\n                "temporal_attention": last_temporal_attention,\n                "valid_event_mask": valid,\n            }\n\n        return logits\n', encoding='utf-8')
(WORK_CODE/'network_dl_dataset_v1_1.py').write_text('from __future__ import annotations\n\nimport csv\nimport json\nimport random\nfrom collections import OrderedDict, defaultdict\nfrom pathlib import Path\nfrom typing import Any, Iterator\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import Dataset, Sampler\n\n\nclass PackedNetworkDataset(Dataset):\n    def __init__(\n        self,\n        dataset_root: str | Path,\n        split: str,\n        *,\n        cache_shards: int = 2\n    ):\n        self.dataset_root = Path(dataset_root)\n        self.split = split.lower().strip()\n        self.cache_shards = max(1, int(cache_shards))\n        self.app_family_col = None\n\n        if self.split not in {"train", "val", "test"}:\n            raise ValueError("split must be train/val/test")\n\n        manifest = (\n            self.dataset_root\n            / "split"\n            / "network_sample_manifest.csv"\n        )\n        if not manifest.exists():\n            raise FileNotFoundError(manifest)\n\n        self.rows: list[dict[str, Any]] = []\n\n        with manifest.open(\n            "r",\n            newline="",\n            encoding="utf-8",\n        ) as f:\n            for row in csv.DictReader(f):\n                if row["split"] != self.split:\n                    continue\n\n                self.rows.append({\n                    "sample_id": row["sample_id"],\n                    "split": row["split"],\n                    "date": row["date"],\n                    "hostname": row["hostname"],\n                    "window_start_epoch": int(row["window_start_epoch"]),\n                    "chunk_index": int(row["chunk_index"]),\n                    "label": int(row["label"]),\n                    "event_count": int(row["event_count"]),\n                    "positive_event_count": int(\n                        row["positive_event_count"]\n                    ),\n                    "shard_file": row["shard_file"],\n                    "shard_sample_index": int(\n                        row["shard_sample_index"]\n                    ),\n                })\n\n        if not self.rows:\n            raise RuntimeError(\n                f"No samples found for split={self.split}"\n            )\n\n        self._cache: OrderedDict[str, dict[str, np.ndarray]] = OrderedDict()\n\n    def __len__(self) -> int:\n        return len(self.rows)\n\n    def _load_shard(\n        self,\n        relative_path: str,\n    ) -> dict[str, np.ndarray]:\n        cached = self._cache.get(relative_path)\n        if cached is not None:\n            self._cache.move_to_end(relative_path)\n            return cached\n\n        path = self.dataset_root / relative_path\n        if not path.exists():\n            raise FileNotFoundError(path)\n\n        with np.load(path, allow_pickle=False) as z:\n            shard = {key: z[key] for key in z.files}\n\n        self._cache[relative_path] = shard\n        self._cache.move_to_end(relative_path)\n\n        while len(self._cache) > self.cache_shards:\n            self._cache.popitem(last=False)\n\n        return shard\n\n    def _x_cat(self, x):\n        if self.app_family_col is None:\n            return x\n        x = x.copy()  # shard cache giu nguyen, chi sua ban sao\n        x[:, self.app_family_col] = 0\n        return x\n\n    def __getitem__(self, index: int) -> dict[str, Any]:\n        row = self.rows[index]\n        shard = self._load_shard(row["shard_file"])\n        i = row["shard_sample_index"]\n\n        offsets = shard["sample_offsets"]\n        start = int(offsets[i])\n        end = int(offsets[i + 1])\n\n        if end - start != row["event_count"]:\n            raise RuntimeError(\n                f"Manifest/shard event_count mismatch: {row[\'sample_id\']}"\n            )\n\n        return {\n            "sample_id": row["sample_id"],\n            "label": int(shard["sample_labels"][i]),\n            "event_count": end - start,\n            "x_cat": self._x_cat(shard["x_cat"][start:end]),\n            "x_num": shard["x_num"][start:end],\n            "x_num_missing": shard["x_num_missing"][start:end],\n            "x_bin": shard["x_bin"][start:end],\n            "delta_t_log1p": shard["delta_t_log1p"][start:end],\n        }\n\n\nclass ShardAwareBatchSampler(Sampler[list[int]]):\n    def __init__(\n        self,\n        dataset: PackedNetworkDataset,\n        batch_size: int,\n        *,\n        shuffle: bool,\n        seed: int = 42,\n        drop_last: bool = False,\n    ):\n        self.dataset = dataset\n        self.batch_size = int(batch_size)\n        self.shuffle = bool(shuffle)\n        self.seed = int(seed)\n        self.drop_last = bool(drop_last)\n        self.epoch = 0\n\n        grouped: dict[str, list[int]] = defaultdict(list)\n        for idx, row in enumerate(dataset.rows):\n            grouped[row["shard_file"]].append(idx)\n\n        self.groups = list(grouped.items())\n\n        total = 0\n        for _shard, indices in self.groups:\n            if drop_last:\n                total += len(indices) // self.batch_size\n            else:\n                total += (len(indices) + self.batch_size - 1) // self.batch_size\n        self._len = total\n\n    def set_epoch(self, epoch: int) -> None:\n        self.epoch = int(epoch)\n\n    def __len__(self) -> int:\n        return self._len\n\n    def __iter__(self) -> Iterator[list[int]]:\n        rng = random.Random(self.seed + self.epoch)\n\n        groups = [\n            (shard, list(indices))\n            for shard, indices in self.groups\n        ]\n\n        if self.shuffle:\n            rng.shuffle(groups)\n\n        for _shard, indices in groups:\n            if self.shuffle:\n                rng.shuffle(indices)\n\n            for start in range(0, len(indices), self.batch_size):\n                batch = indices[start:start + self.batch_size]\n\n                if self.drop_last and len(batch) < self.batch_size:\n                    continue\n\n                yield batch\n\n\ndef collate_sequence_batch(\n    batch: list[dict[str, Any]],\n) -> dict[str, Any]:\n    if not batch:\n        raise ValueError("Empty batch")\n\n    lengths = torch.tensor(\n        [x["event_count"] for x in batch],\n        dtype=torch.long,\n    )\n\n    max_len = int(lengths.max().item())\n    bsz = len(batch)\n    c = batch[0]["x_cat"].shape[1]\n    n = batch[0]["x_num"].shape[1]\n    binary_dim = batch[0]["x_bin"].shape[1]\n\n    x_cat = torch.zeros(bsz, max_len, c, dtype=torch.long)\n    x_num = torch.zeros(bsz, max_len, n, dtype=torch.float32)\n    x_missing = torch.zeros(bsz, max_len, n, dtype=torch.float32)\n    x_bin = torch.zeros(bsz, max_len, binary_dim, dtype=torch.float32)\n    delta = torch.zeros(bsz, max_len, 1, dtype=torch.float32)\n    mask = torch.zeros(bsz, max_len, dtype=torch.bool)\n    labels = torch.tensor(\n        [x["label"] for x in batch],\n        dtype=torch.float32,\n    )\n\n    for i, item in enumerate(batch):\n        t = item["event_count"]\n\n        x_cat[i, :t] = torch.from_numpy(\n            np.asarray(item["x_cat"], dtype=np.int64)\n        )\n        x_num[i, :t] = torch.from_numpy(\n            np.asarray(item["x_num"], dtype=np.float32)\n        )\n        x_missing[i, :t] = torch.from_numpy(\n            np.asarray(item["x_num_missing"], dtype=np.float32)\n        )\n        x_bin[i, :t] = torch.from_numpy(\n            np.asarray(item["x_bin"], dtype=np.float32)\n        )\n        delta[i, :t, 0] = torch.from_numpy(\n            np.asarray(item["delta_t_log1p"], dtype=np.float32)\n        )\n        mask[i, :t] = True\n\n    return {\n        "x_cat": x_cat,\n        "x_num": x_num,\n        "x_num_missing": x_missing,\n        "x_bin": x_bin,\n        "delta_t_log1p": delta,\n        "attention_mask": mask,\n        "labels": labels,\n        "lengths": lengths,\n        "sample_ids": [x["sample_id"] for x in batch],\n    }\n', encoding='utf-8')
print('Wrote:', sorted(p.name for p in WORK_CODE.iterdir()))
sys.path.insert(0, str(WORK_CODE))

## 3. Forward smoke test

Chạy một batch nhỏ để xác nhận shape, model và GPU memory trước khi full training.


In [ ]:
from network_dl_dataset_v1_1 import PackedNetworkDataset, ShardAwareBatchSampler, collate_sequence_batch
from mixed_type_attention_model import MixedTypeModelConfig, MixedTypeSelfAttentionClassifier
from torch.utils.data import DataLoader

spec = json.loads((DATASET_ROOT/'preprocessing/network_preprocess_spec.json').read_text())
maps = json.loads((DATASET_ROOT/'preprocessing/network_category_maps.json').read_text())
cat_fields = spec['active_categorical_features']
num_fields = spec['active_numerical_features']
cards = [max(int(v) for v in maps['fields'][f]['mapping'].values()) + 1 for f in cat_fields]

ds = PackedNetworkDataset(DATASET_ROOT, 'train', cache_shards=1)
sampler = ShardAwareBatchSampler(ds, 4, shuffle=False)
loader = DataLoader(ds, batch_sampler=sampler, num_workers=0, collate_fn=collate_sequence_batch)
batch = next(iter(loader))
print('app_family ids in batch:', sorted(set(batch['x_cat'][:, :, 0].unique().tolist()))[:10])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg = MixedTypeModelConfig(
    categorical_cardinalities=cards,
    numerical_dim=len(num_fields),
    max_events=512,
    d_model=64,
    num_heads=4,
    feature_layers=1,
    temporal_layers=2,
    ff_dim=128,
    dropout=0.2,
)
model = MixedTypeSelfAttentionClassifier(cfg).to(device)

with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type=='cuda'):
    logits = model(
        x_cat=batch['x_cat'].to(device),
        x_num=batch['x_num'].to(device),
        x_num_missing=batch['x_num_missing'].to(device),
        x_bin=batch['x_bin'].to(device),
        delta_t_log1p=batch['delta_t_log1p'].to(device),
        attention_mask=batch['attention_mask'].to(device),
    )

print('logits:', tuple(logits.shape))
print('params:', sum(p.numel() for p in model.parameters() if p.requires_grad))
if device.type == 'cuda':
    print('allocated GB:', torch.cuda.memory_allocated()/1024**3)
del model, batch
if device.type == 'cuda':
    torch.cuda.empty_cache()


## 4. Full training

Default cho T4: physical batch 16 + gradient accumulation 2, effective batch ≈ 32. VAL/TEST sẽ in progress mỗi 100 batches.


In [ ]:
OUTPUT_DIR = Path(f'/kaggle/working/{RUN_NAME}')
cmd = (
    f"python {WORK_CODE/'05e_train_mixed_type_attention.py'} "
    f"--dataset-root '{DATASET_ROOT}' "
    f"--output-dir '{OUTPUT_DIR}' "
    "--epochs 8 "
    "--batch-size 16 "
    "--grad-accum 2 "
    "--d-model 64 "
    "--heads 4 "
    "--feature-layers 1 "
    "--temporal-layers 2 "
    "--ff-dim 128 "
    "--dropout 0.2 "
    "--lr 0.0003 "
    "--weight-decay 0.0001 "
    "--grad-clip 1.0 "
    "--patience 3 "
    "--num-workers 2 "
    "--eval-progress-every 100"
)
print(cmd)
import subprocess
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f'Training failed with exit code {ret}')


## 5. Inspect result


In [ ]:
metrics = json.loads((OUTPUT_DIR/'metrics.json').read_text())
threshold = json.loads((OUTPUT_DIR/'threshold.json').read_text())
print(json.dumps(metrics, indent=2))
print('\nThreshold:')
print(json.dumps(threshold, indent=2))


## 6. Attention audit

Chỉ dùng như diagnostic. Attention weight không đồng nghĩa causal feature importance.


In [ ]:
cmd = (
    f"python {WORK_CODE/'05e_attention_audit.py'} "
    f"--dataset-root '{DATASET_ROOT}' "
    f"--model-dir '{OUTPUT_DIR}' "
    "--split val "
    "--positive-samples 8 "
    "--negative-samples 8"
)
print(cmd)
import subprocess
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f'Attention audit failed with exit code {ret}')
audit = json.loads((OUTPUT_DIR/'attention_audit.json').read_text())
print(json.dumps(audit['feature_attention_received'], indent=2)[:7000])


## 7. Zip outputs


In [ ]:
archive = shutil.make_archive(
    f'/kaggle/working/{RUN_NAME}',
    'zip',
    OUTPUT_DIR,
)
print('Created:', archive)
